# Clasificacion de urgencia con embeddings multilingues + Logistic Regression.
Mismos datos y mismo split que 02_baseline.py, para comparar de forma justa.

In [1]:
import os
import joblib
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

load_dotenv()

MODELO_EMBEDDINGS = "paraphrase-multilingual-mpnet-base-v2"
VERSION = "v1"

/home/jorge/proyectos/sicpc-thesis/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### 1. Cargar datos (identico al baseline)

In [2]:
os.chdir("..")  # sube un nivel, a la raiz del proyecto
print(os.getcwd())
df_kaggle = pd.read_csv("data/processed/kaggle_es_baseline.csv")
df_kaggle["texto"] = df_kaggle["subject"].fillna("") + " " + df_kaggle["body"].fillna("")
df_kaggle = df_kaggle[["texto", "urgencia"]]
df_kaggle["fuente"] = "kaggle"

df_reales = pd.DataFrame(columns=["texto", "urgencia", "fuente"])
try:
    import mysql.connector

    conn = mysql.connector.connect(
        host="localhost", port=3306,
        user=os.getenv("DB_USER", "sJorge"), password=os.getenv("DB_PASSWORD"),
        database="sicpc",
    )
    df_reales = pd.read_sql(
        "SELECT asunto_anonimizado, cuerpo_anonimizado, urgencia_real "
        "FROM correo WHERE urgencia_real IS NOT NULL",
        conn,
    )
    conn.close()

    if len(df_reales) > 0:
        df_reales["texto"] = (
            df_reales["asunto_anonimizado"].fillna("") + " " +
            df_reales["cuerpo_anonimizado"].fillna("")
        )
        df_reales = df_reales.rename(columns={"urgencia_real": "urgencia"})
        df_reales = df_reales[["texto", "urgencia"]]
        df_reales["fuente"] = "real"
except Exception as e:
    print(f"No se pudieron cargar correos reales: {e}")

df = pd.concat([df_kaggle, df_reales], ignore_index=True)
df = df.dropna(subset=["texto", "urgencia"])
df = df[df["texto"].str.strip() != ""]

print(f"Total: {len(df)} filas")
print(df["fuente"].value_counts())


/home/jorge/proyectos/sicpc-thesis
Total: 1012 filas
fuente
kaggle    812
real      200
Name: count, dtype: int64


/tmp/ipykernel_56510/3444991047.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_reales = pd.read_sql(


#### 2. Split (mismo que el baseline)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df["texto"], df["urgencia"],
    test_size=0.2, random_state=42, stratify=df["urgencia"],
)

#### 3. Embeddings (el modelo NO se reentrena)

In [4]:
print(f"\nCargando modelo de embeddings: {MODELO_EMBEDDINGS}")
encoder = SentenceTransformer(MODELO_EMBEDDINGS)

print("Generando embeddings de entrenamiento...")
X_train_emb = encoder.encode(
    X_train.tolist(), batch_size=32, show_progress_bar=True,
    normalize_embeddings=True,
)
print("Generando embeddings de prueba...")
X_test_emb = encoder.encode(
    X_test.tolist(), batch_size=32, show_progress_bar=True,
    normalize_embeddings=True,
)


Cargando modelo de embeddings: paraphrase-multilingual-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2187.80it/s]


Generando embeddings de entrenamiento...


Batches: 100%|██████████| 26/26 [02:03<00:00,  4.77s/it]


Generando embeddings de prueba...


Batches: 100%|██████████| 7/7 [00:28<00:00,  4.08s/it]


#### 4. Clasificador

In [5]:
modelo = LogisticRegression(max_iter=2000, class_weight="balanced")
modelo.fit(X_train_emb, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


#### 5. Evaluacion

In [6]:
y_pred = modelo.predict(X_test_emb)

print("\n" + "=" * 60)
print("REPORTE DE CLASIFICACION (embeddings)")
print("=" * 60)
print(classification_report(y_test, y_pred))

print("MATRIZ DE CONFUSION")
print(confusion_matrix(y_test, y_pred, labels=["alta", "media", "baja"]))
print("(orden de filas/columnas: alta, media, baja)")

# Metricas por fuente (Kaggle vs correos reales)
fuente_test = df.loc[X_test.index, "fuente"]
for f in ["kaggle", "real"]:
    mask = (fuente_test == f).values
    if mask.sum() > 0:
        print(f"\n--- Solo {f} (n={mask.sum()}) ---")
        print(classification_report(y_test[mask], y_pred[mask]))


REPORTE DE CLASIFICACION (embeddings)
              precision    recall  f1-score   support

        alta       0.70      0.72      0.71        79
        baja       0.59      0.44      0.51        54
       media       0.49      0.56      0.52        70

    accuracy                           0.59       203
   macro avg       0.59      0.57      0.58       203
weighted avg       0.59      0.59      0.59       203

MATRIZ DE CONFUSION
[[57 17  5]
 [19 39 12]
 [ 6 24 24]]
(orden de filas/columnas: alta, media, baja)

--- Solo kaggle (n=171) ---
              precision    recall  f1-score   support

        alta       0.68      0.73      0.71        67
        baja       0.48      0.26      0.33        39
       media       0.47      0.57      0.52        65

    accuracy                           0.56       171
   macro avg       0.54      0.52      0.52       171
weighted avg       0.56      0.56      0.55       171


--- Solo real (n=32) ---
              precision    recall  f1-scor

#### 6. Guardar clasificador + nombre del encoder (van siempre juntos)

In [7]:
os.makedirs("models", exist_ok=True)
joblib.dump(
    {"clasificador": modelo, "encoder": MODELO_EMBEDDINGS},
    f"models/embeddings_{VERSION}.pkl",
)
print(f"\nGuardado en models/embeddings_{VERSION}.pkl")


Guardado en models/embeddings_v1.pkl


In [8]:
res = pd.DataFrame({"texto": X_test.values, "real": y_test.values,
                    "pred": y_pred, "fuente": fuente_test.values})
errores = res[(res["fuente"] == "real") & (res["real"] != res["pred"])]
for _, r in errores.iterrows():
    print(f"REAL={r['real']} PRED={r['pred']} | {r['texto'][:200]}\n")

REAL=alta PRED=baja | El saldo total a debitar es $155.562,51 y vence el 05/11/2026 
 

[DIRECCION] 
VER ONLINE  


 

 
body {margin: 0;}
    /* Estilos Específicos del Hero */
    .hero-text {
 
font-family

REAL=media PRED=baja | Tu  
Hola:


Ingresá este código para continuar con el inicio de sesión sin contraseña:


834361


Este código es válido por 20 minutos y solo se puede usar una vez. Al ingresarlo, también v

REAL=baja PRED=alta | Alerta meteorológica por lluvias y vientos fuertes 
											QUÉ HACER 
											 ANTE LLUVIAS Y VIENTOS FUERTES
										
											[DIRECCION] limpias las rejillas
										
											[PERSO

REAL=alta PRED=baja | Asignación de turnos  
                    
                        
                            [DIRECCION]
                            [DIRECCION] 1068
                            
           

REAL=media PRED=alta | Issue 353: There's no need to panic The SaaSPocalpyse, multiplayer AI, and jailbreaking your console
The Overf

#### Verifico si el kernel sigue en memoria con los datos, si da Error ejecutar todo nuevamente

In [9]:
print(len(df), len(y_pred))

1012 203


##### Celda A. Muestra cómo se distribuyen las predicciones en el test de correos reales y lista los errores.

In [10]:
fuente_test = df.loc[X_test.index, "fuente"]
res = pd.DataFrame({
    "texto": X_test.values,
    "real": y_test.values,
    "pred": y_pred,
    "fuente": fuente_test.values,
})
reales = res[res["fuente"] == "real"]

print(f"Test de correos reales (n={len(reales)})\n")
print(pd.DataFrame({
    "verdad": reales["real"].value_counts(),
    "predicho": reales["pred"].value_counts(),
}).fillna(0).astype(int))

print("\nMatriz de confusión (filas = real, columnas = predicho)")
print(pd.crosstab(reales["real"], reales["pred"]))

errores = reales[reales["real"] != reales["pred"]]
print(f"\nErrores: {len(errores)} de {len(reales)}\n")
for _, r in errores.iterrows():
    texto = " ".join(r["texto"].split())[:200]
    print(f"REAL={r['real']} | PRED={r['pred']} | {texto}\n")

Test de correos reales (n=32)

       verdad  predicho
baja       15        20
alta       12        10
media       5         2

Matriz de confusión (filas = real, columnas = predicho)
pred   alta  baja  media
real                    
alta      8     4      0
baja      1    14      0
media     1     2      2

Errores: 8 de 32

REAL=alta | PRED=baja | El saldo total a debitar es $155.562,51 y vence el 05/11/2026 [DIRECCION] VER ONLINE body {margin: 0;} /* Estilos Específicos del Hero */ .hero-text { font-family: 'Roboto', Arial, sans-serif; font-si

REAL=media | PRED=baja | Tu Hola: Ingresá este código para continuar con el inicio de sesión sin contraseña: 834361 Este código es válido por 20 minutos y solo se puede usar una vez. Al ingresarlo, también vas a confirmar la 

REAL=baja | PRED=alta | Alerta meteorológica por lluvias y vientos fuertes QUÉ HACER ANTE LLUVIAS Y VIENTOS FUERTES [DIRECCION] limpias las rejillas [PERSONA] toldos y persianas Retirá macetas y elementos que puedan cae

##### Celda B. Mide si los correos de urgencia alta traen palabras explícitas de urgencia, comparando Kaggle contra los correos reales sobre todos los datos, no solo el test.

In [11]:
SENALES = r"urgente|urgencia|hoy|cuanto antes|lo antes posible|a la brevedad|inmediat|vence|vencimiento|plazo|asap|cr[ií]tico"
df["tiene_senal"] = df["texto"].str.contains(SENALES, case=False, regex=True)

print(df.groupby(["fuente", "urgencia"])["tiene_senal"]
        .agg(["mean", "count"]).round(2))

                 mean  count
fuente urgencia             
kaggle alta      0.74    338
       baja      0.23    159
       media     0.39    315
real   alta      0.11     54
       baja      0.07    110
       media     0.19     36
